# Load the data into PostgreSQL, MySQL, SQL Server or SQLite

The same dataset can be written as a script for each database. This notebook generates one dataset in four dialects, shows how the scripts differ, and loads the SQLite one.

**Uses:** 12,000 records (3,000 per dialect), no AI-generated fields. API reference: [generaterealdata.com/docs/api/](https://generaterealdata.com/docs/api/).

## Setup

You need an API key (PRO, ULTRA or Enterprise plan): sign in at [generaterealdata.com](https://generaterealdata.com), open the dashboard and click **Generate API key**. Keep it out of the notebook: put it in the `GRD_API_KEY` environment variable, or in Colab's **Secrets** panel under the same name.

In [1]:
import os
import time
from pathlib import Path

import requests

API = "https://generaterealdata.com/api/v1"
KEY = os.environ.get("GRD_API_KEY")
if not KEY:                                   # Google Colab: read it from the Secrets panel
    from google.colab import userdata
    KEY = userdata.get("GRD_API_KEY")
HEADERS = {"Authorization": f"Bearer {KEY}"}
OUT = Path("downloads")
OUT.mkdir(exist_ok=True)


def api(method, path, **kwargs):
    """One API call. Waits and tries again if the API says to slow down or another dataset is still running."""
    while True:
        r = requests.request(method, API + path, headers=HEADERS, timeout=90, **kwargs)
        busy = r.status_code == 409 and r.json().get("error", {}).get("code") == "run_in_progress"
        if r.status_code == 429 or busy:
            time.sleep(int(r.headers.get("Retry-After", 15)))
            continue
        data = r.json()
        if r.status_code >= 400:
            e = data["error"]
            raise RuntimeError(f"{e['code']}: {e['message']} {e.get('hint', '')}")
        return data


def generate(spec):
    """Start a dataset and wait until it is ready. Returns its final status."""
    ds = api("POST", "/datasets", json=spec)
    while ds["status"] not in ("ready", "failed"):
        time.sleep(5)
        ds = api("GET", f"/datasets/{ds['id']}")
    if ds["status"] == "failed":
        raise RuntimeError(ds["error"]["message"])
    return ds


def download(ds, filename, file=None):
    """Save a ready dataset's file. file="db" fetches the SQLite file of an SQL dataset."""
    path = OUT / filename
    url = f"{API}/datasets/{ds['id']}/download"
    with requests.get(url, params={"file": file} if file else None, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return path


def summary(ds):
    """The facts worth reading from a dataset's status."""
    keys = ("status", "format", "rows_generated", "rows_per_table", "credits_charged", "ai_rows_charged", "schema_id", "seed")
    return {k: ds[k] for k in keys if k in ds}

## 1. One schema, four dialects

The first run designs the schema. The other three reuse its `schema_id` with the same `seed`, so all four scripts hold the same rows.

In [2]:
def repo_file(path):
    """Read a file of this repository: from disk when the repo is cloned, from GitHub otherwise (e.g. in Colab)."""
    local = Path("..") / path
    if local.exists():
        return local.read_text(encoding="utf-8")
    r = requests.get("https://raw.githubusercontent.com/generaterealdata/api-examples/main/" + path, timeout=30)
    r.raise_for_status()
    return r.text


SHOP_SQL = repo_file("schemas/shop.sql")
SHOP_RULES = repo_file("schemas/shop_rules.txt").strip()

first = generate({"mode": "sql", "sql": SHOP_SQL, "notes": SHOP_RULES, "rows": 3000,
                  "format": "sql", "dialect": "postgresql", "seed": 4})
scripts = {"postgresql": download(first, "shop_postgresql.sql")}
for dialect in ("mysql", "sqlserver", "sqlite"):
    ds = generate({"mode": "schema", "schema_id": first["schema_id"], "rows": 3000,
                   "format": "sql", "dialect": dialect, "seed": 4})
    scripts[dialect] = download(ds, f"shop_{dialect}.sql")
{d: f"{p.stat().st_size / 1024:.0f} KB" for d, p in scripts.items()}

{'postgresql': '138 KB',
 'mysql': '138 KB',
 'sqlserver': '138 KB',
 'sqlite': '137 KB'}

## 2. How the scripts differ

Each script uses its database's own types, quoting and syntax. Here is the `orders` table in each.

In [3]:
import re

for dialect, path in scripts.items():
    text = path.read_text(encoding="utf-8")
    table = re.search(r"CREATE TABLE[^\n]*orders[^\n]*\n.*?\n\)[^\n]*", text, re.S)
    print(f"-- {dialect}\n{table.group(0)}\n")

-- postgresql
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER       NOT NULL REFERENCES customers(customer_id),
    order_date  DATE          NOT NULL,
    status      VARCHAR(12)   NOT NULL CHECK (status IN ('pending', 'shipped', 'delivered', 'cancelled')),
    order_total NUMERIC(12,2) NOT NULL
);

-- mysql
CREATE TABLE `orders` (
    `order_id` INT NOT NULL,
    `customer_id` INT NOT NULL,
    `order_date` DATE NOT NULL,
    `status` VARCHAR(12) NOT NULL CHECK (`status` IN ('pending', 'shipped', 'delivered', 'cancelled')),
    `order_total` DECIMAL(12,2) NOT NULL,
    PRIMARY KEY (`order_id`)
);

-- sqlserver
CREATE TABLE [orders] (
    [order_id] INT NOT NULL,
    [customer_id] INT NOT NULL,
    [order_date] DATE NOT NULL,
    [status] NVARCHAR(12) NOT NULL CHECK ([status] IN ('pending', 'shipped', 'delivered', 'cancelled')),
    [order_total] DECIMAL(12,2) NOT NULL,
    PRIMARY KEY ([order_id])
);

-- sqlite
CREATE TABLE "orders" (
    "order_id"

## 3. Load it

| Database | Command |
|---|---|
| PostgreSQL | `psql -d your_database -f shop_postgresql.sql` |
| MySQL | `mysql your_database < shop_mysql.sql` |
| SQL Server | `sqlcmd -d your_database -i shop_sqlserver.sql` |
| SQLite | `sqlite3 shop.db < shop_sqlite.sql` |

Each script creates the tables and inserts the rows in foreign-key order. To add the data to tables that already exist, send `"include_schema": false`; to replace them, send `"drop_existing": true`.

SQLite ships with Python, so that one can be loaded right here:

In [4]:
import sqlite3

import pandas as pd

target = OUT / "shop_loaded.db"
target.unlink(missing_ok=True)
con = sqlite3.connect(target)
con.executescript(scripts["sqlite"].read_text(encoding="utf-8"))
pd.DataFrame([(t, con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0])
              for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'")],
             columns=["table", "rows loaded"])

,table,rows loaded
0,customers,294
1,products,59
2,orders,882
3,order_items,1765


The database enforces the keys from now on. SQLite's own integrity check finds no row that points at a missing parent:

In [5]:
con.execute("PRAGMA foreign_keys = ON")
print("Foreign-key violations:", len(con.execute("PRAGMA foreign_key_check").fetchall()))

Foreign-key violations: 0


## 4. Skip the script

Every `format: "sql"` dataset also comes as a ready-made SQLite file: `download(ds, "shop.sqlite", file="db")`. And if you load with your own tools, `format: "csv"` gives one CSV per table.